# 参数拦截与前置纠错

## 学习重点
编写专门的"参数修复 Prompt"或在 Python 后端建立静态硬编码映射。

## 核心痛点
大模型调用工具时极易发生"幻觉"（比如编造一个不存在的参数名）。在工具真正执行前，必须有一层代码对其进行严格的语法与语义检查。

## 0. 环境准备

虚拟环境放在项目根目录下，所有课程共享同一个 venv。

在终端中执行以下命令（**不要在 Notebook 中运行**）：

```bash
cd ~/Documents/ObsidianVault/CS/KnowledgeBase
source .venv/bin/activate
uv pip install pydantic
```

然后在 Jupyter Notebook 中选择 Kernel 为 **KnowledgeBase (.venv)** 即可。

In [ ]:
from pydantic import BaseModel, Field, field_validator, model_validator
from pydantic import ValidationError
from typing import Optional, Literal, Any
from datetime import datetime, date
import json
import re

---
## 1. field_validator vs model_validator — 两种拦截模式对比

| 模式 | `field_validator` 接收的 v | `model_validator` 接收的 v |
|------|--------------------------|--------------------------|
| `mode='before'` | 该字段的原始输入值 | 整个模型的**原始输入字典（dict）** |
| `mode='after'` | 该字段解析后的安全值 | 已实例化的 **Model 对象本身（self）** |
| `mode='wrap'` | 原始输入值 + `handler` | 原始输入字典（dict）+ `handler` |

如果校验逻辑**同时牵扯多个字段**（字段之间有联动、联合制约关系），`field_validator` 搞不定，必须用 `model_validator`。

In [ ]:
class DemoFieldBefore(BaseModel):
    value: int

    @field_validator('value', mode='before')
    @classmethod
    def before_check(cls, v):
        print(f"  [field before] v = {v!r} (type={type(v).__name__})")
        return v

class DemoModelBefore(BaseModel):
    x: int
    y: int

    @model_validator(mode='before')
    @classmethod
    def before_check(cls, v):
        print(f"  [model before] v = {v!r} (type={type(v).__name__})")
        return v

print("--- field_validator mode='before' ---")
DemoFieldBefore(value="42")

print("\n--- model_validator mode='before' ---")
DemoModelBefore(x=1, y=2)

In [ ]:
class DemoFieldAfter(BaseModel):
    value: int

    @field_validator('value', mode='after')
    @classmethod
    def after_check(cls, v):
        print(f"  [field after] v = {v!r} (type={type(v).__name__})")
        return v

class DemoModelAfter(BaseModel):
    x: int
    y: int

    @model_validator(mode='after')
    def after_check(self):
        print(f"  [model after] self.x={self.x}, self.y={self.y}")
        return self

print("--- field_validator mode='after' ---")
DemoFieldAfter(value="42")

print("\n--- model_validator mode='after' ---")
DemoModelAfter(x=1, y=2)

---
## 2. 参数拦截 — 类型检查与范围验证

### 2.1 类型清洗：大模型传错类型的拦截

大模型常见幻觉：把字符串 ID 传成数字、把枚举值传成同义词。

In [ ]:
class DatabaseQuery(BaseModel):
    table_name: str
    limit: int = Field(default=10, ge=1, le=1000)
    offset: int = Field(default=0, ge=0)

    @field_validator('table_name', mode='before')
    @classmethod
    def coerce_table_name(cls, v):
        if isinstance(v, int):
            v = str(v)
        if isinstance(v, str):
            v = v.strip()
            if not re.match(r'^[a-zA-Z_]\w*$', v):
                raise ValueError(f'table_name 不合法: {v!r}，只允许字母/数字/下划线')
            return v
        raise ValueError(f'table_name 类型不合法: {type(v).__name__}')

    @field_validator('limit', mode='before')
    @classmethod
    def coerce_limit(cls, v):
        if isinstance(v, str):
            try:
                return int(v)
            except ValueError:
                raise ValueError(f'limit 无法转为整数: {v!r}')
        return v

q1 = DatabaseQuery(table_name=123, limit="50")
print(f"q1: table_name={q1.table_name!r}, limit={q1.limit}")

q2 = DatabaseQuery(table_name="users", limit=100)
print(f"q2: table_name={q2.table_name!r}, limit={q2.limit}")

In [ ]:
try:
    DatabaseQuery(table_name="drop table users", limit=10)
except ValidationError as e:
    print(f"SQL 注入拦截: {e.errors()[0]['msg']}")

try:
    DatabaseQuery(table_name="users", limit=9999)
except ValidationError as e:
    print(f"范围越界: {e.errors()[0]['msg']}")

### 2.2 枚举值拦截 — 大模型编造不存在的参数

大模型幻觉重灾区：编造一个不存在的枚举值，或者用同义词代替。

In [ ]:
VALID_CITIES = ["北京", "上海", "广州", "深圳", "杭州", "成都"]

CITY_ALIASES = {
    "北京": ["Beijing", "beijing", "BJ", "bj", "帝都"],
    "上海": ["Shanghai", "shanghai", "SH", "sh", "魔都"],
    "广州": ["Guangzhou", "guangzhou", "GZ", "gz"],
    "深圳": ["Shenzhen", "shenzhen", "SZ", "sz"],
    "杭州": ["Hangzhou", "hangzhou", "HZ", "hz"],
    "成都": ["Chengdu", "chengdu", "CD", "cd"],
}

ALIAS_TO_CANONICAL = {}
for canonical, aliases in CITY_ALIASES.items():
    for alias in aliases:
        ALIAS_TO_CANONICAL[alias] = canonical

class WeatherQuery(BaseModel):
    city: str
    date: str

    @field_validator('city', mode='before')
    @classmethod
    def normalize_city(cls, v):
        if not isinstance(v, str):
            raise ValueError(f'city 必须是字符串，收到 {type(v).__name__}')
        v = v.strip()
        if v in VALID_CITIES:
            return v
        if v in ALIAS_TO_CANONICAL:
            return ALIAS_TO_CANONICAL[v]
        raise ValueError(f'不支持的城市: {v!r}，支持: {", ".join(VALID_CITIES)}')

    @field_validator('date', mode='before')
    @classmethod
    def validate_date(cls, v):
        if isinstance(v, str):
            for fmt in ('%Y-%m-%d', '%Y/%m/%d', '%Y年%m月%d日'):
                try:
                    datetime.strptime(v, fmt)
                    return datetime.strptime(v, fmt).strftime('%Y-%m-%d')
                except ValueError:
                    continue
            raise ValueError(f'日期格式错误: {v!r}')
        raise ValueError(f'日期类型错误: {type(v).__name__}')

w1 = WeatherQuery(city="Beijing", date="2024-01-15")
print(f"w1: city={w1.city!r}, date={w1.date}")

w2 = WeatherQuery(city="魔都", date="2024/03/20")
print(f"w2: city={w2.city!r}, date={w2.date}")

w3 = WeatherQuery(city="BJ", date="2024年12月25日")
print(f"w3: city={w3.city!r}, date={w3.date}")

In [ ]:
try:
    WeatherQuery(city="纽约", date="2024-01-15")
except ValidationError as e:
    print(f"不支持的城市: {e.errors()[0]['msg']}")

try:
    WeatherQuery(city="北京", date="Jan 15 2024")
except ValidationError as e:
    print(f"日期格式错误: {e.errors()[0]['msg']}")

---
## 3. 前置纠错 — 静态硬编码映射

### 3.1 参数名纠错

大模型经常编造不存在的参数名，比如把 `city` 写成 `location`，把 `date` 写成 `time`。我们可以用硬编码映射自动纠错。

In [ ]:
class ParamFixer:
    def __init__(self, alias_map: dict[str, str], valid_params: set[str]):
        self.alias_map = alias_map
        self.valid_params = valid_params

    def fix(self, params: dict) -> dict:
        fixed = {}
        for key, value in params.items():
            if key in self.valid_params:
                fixed[key] = value
            elif key in self.alias_map:
                canonical = self.alias_map[key]
                print(f"  [纠错] 参数名 '{key}' → '{canonical}'")
                fixed[canonical] = value
            else:
                print(f"  [拦截] 未知参数 '{key}'，已丢弃")
        return fixed

weather_fixer = ParamFixer(
    alias_map={
        "location": "city",
        "place": "city",
        "region": "city",
        "time": "date",
        "when": "date",
        "day": "date",
    },
    valid_params={"city", "date"}
)

llm_output = {
    "location": "北京",
    "time": "2024-01-15",
    "temperature_unit": "celsius",
}

print("原始输入:", llm_output)
fixed = weather_fixer.fix(llm_output)
print("纠错后:", fixed)

query = WeatherQuery(**fixed)
print(f"最终结果: city={query.city!r}, date={query.date}")

### 3.2 参数值纠错 — 模糊匹配

大模型可能把参数值写错一点点，比如 `celsiu` 而不是 `celsius`，我们可以用模糊匹配自动纠错。

In [ ]:
from difflib import get_close_matches

class FuzzyParamFixer:
    def __init__(self, valid_values: dict[str, list[str]]):
        self.valid_values = valid_values

    def fix_value(self, param_name: str, value: str, cutoff: float = 0.6) -> str:
        if param_name not in self.valid_values:
            return value
        valid = self.valid_values[param_name]
        if value in valid:
            return value
        matches = get_close_matches(value, valid, n=1, cutoff=cutoff)
        if matches:
            print(f"  [模糊纠错] {param_name}='{value}' → '{matches[0]}'")
            return matches[0]
        raise ValueError(f'{param_name} 的值 \"{value}\" 无法纠错，合法值: {valid}')

    def fix_dict(self, params: dict) -> dict:
        fixed = {}
        for key, value in params.items():
            if isinstance(value, str) and key in self.valid_values:
                fixed[key] = self.fix_value(key, value)
            else:
                fixed[key] = value
        return fixed

fuzzy_fixer = FuzzyParamFixer(valid_values={
    "unit": ["celsius", "fahrenheit"],
    "method": ["GET", "POST", "PUT", "DELETE"],
    "level": ["INFO", "WARN", "ERROR", "DEBUG"],
})

test_cases = [
    {"unit": "celsiu", "method": "POST"},
    {"unit": "farenheit", "method": "GT"},
    {"unit": "celsius", "method": "PST"},
    {"level": "WAR", "method": "DELETE"},
]

for tc in test_cases:
    print(f"\n输入: {tc}")
    result = fuzzy_fixer.fix_dict(tc)
    print(f"纠错后: {result}")

In [ ]:
try:
    fuzzy_fixer.fix_dict({"unit": "kelvin"})
except ValueError as e:
    print(f"无法纠错: {e}")

---
## 4. 跨字段校验 — model_validator 实战

当字段之间有联动关系时，`field_validator` 无法访问其他字段的值，必须用 `model_validator`。

In [ ]:
class FileOperation(BaseModel):
    operation: Literal["read", "write", "delete"]
    path: str
    content: Optional[str] = None
    encoding: str = "utf-8"

    @field_validator('path', mode='before')
    @classmethod
    def validate_path(cls, v):
        if not isinstance(v, str):
            raise ValueError(f'path 必须是字符串')
        v = v.strip()
        if '..' in v.split('/'):
            raise ValueError(f'路径不允许包含 .. : {v!r}')
        if not v.startswith('/'):
            v = '/' + v
        return v

    @model_validator(mode='after')
    def check_operation_consistency(self):
        if self.operation == 'write' and not self.content:
            raise ValueError('write 操作必须提供 content')
        if self.operation in ('read', 'delete') and self.content:
            raise ValueError(f'{self.operation} 操作不应包含 content')
        return self

ok1 = FileOperation(operation="read", path="/home/user/data.txt")
print(f"read: path={ok1.path}")

ok2 = FileOperation(operation="write", path="/home/user/data.txt", content="hello")
print(f"write: path={ok2.path}, content={ok2.content!r}")

In [ ]:
try:
    FileOperation(operation="write", path="/home/user/data.txt")
except ValidationError as e:
    print(f"write 缺少 content: {e.errors()[0]['msg']}")

try:
    FileOperation(operation="read", path="/home/user/data.txt", content="extra")
except ValidationError as e:
    print(f"read 不应有 content: {e.errors()[0]['msg']}")

try:
    FileOperation(operation="read", path="/home/../etc/passwd")
except ValidationError as e:
    print(f"路径穿越拦截: {e.errors()[0]['msg']}")

---
## 5. 完整拦截管线 — 组合所有技术

将参数名纠错、值纠错、类型清洗、跨字段校验组合成一条完整的拦截管线。

In [ ]:
class ToolCallPipeline:
    def __init__(self, param_alias_map: dict, valid_params: set, fuzzy_values: dict):
        self.param_fixer = ParamFixer(param_alias_map, valid_params)
        self.fuzzy_fixer = FuzzyParamFixer(fuzzy_values)

    def preprocess(self, raw_params: dict) -> dict:
        print("[1] 参数名纠错:")
        step1 = self.param_fixer.fix(raw_params)
        print("[2] 参数值模糊纠错:")
        step2 = self.fuzzy_fixer.fix_dict(step1)
        return step2

pipeline = ToolCallPipeline(
    param_alias_map={
        "location": "city",
        "place": "city",
        "when": "date",
        "time": "date",
        "unit": "unit",
    },
    valid_params={"city", "date", "unit"},
    fuzzy_values={
        "unit": ["celsius", "fahrenheit"],
    }
)

class SafeWeatherQuery(BaseModel):
    city: str
    date: str
    unit: Literal["celsius", "fahrenheit"] = "celsius"

    @field_validator('city', mode='before')
    @classmethod
    def normalize_city(cls, v):
        if not isinstance(v, str):
            raise ValueError(f'city 必须是字符串')
        v = v.strip()
        if v in VALID_CITIES:
            return v
        if v in ALIAS_TO_CANONICAL:
            return ALIAS_TO_CANONICAL[v]
        raise ValueError(f'不支持的城市: {v!r}')

    @field_validator('date', mode='before')
    @classmethod
    def validate_date(cls, v):
        if isinstance(v, str):
            for fmt in ('%Y-%m-%d', '%Y/%m/%d', '%Y年%m月%d日'):
                try:
                    return datetime.strptime(v, fmt).strftime('%Y-%m-%d')
                except ValueError:
                    continue
            raise ValueError(f'日期格式错误: {v!r}')
        raise ValueError(f'日期类型错误: {type(v).__name__}')

llm_outputs = [
    {"location": "Beijing", "when": "2024-01-15", "unit": "celsiu"},
    {"place": "魔都", "time": "2024/03/20", "extra_param": "should_be_removed"},
    {"city": "SZ", "date": "2024年12月25日", "unit": "farenheit"},
]

for i, raw in enumerate(llm_outputs, 1):
    print(f"\n=== 测试 {i} ===")
    print(f"原始输入: {raw}")
    try:
        preprocessed = pipeline.preprocess(raw)
        query = SafeWeatherQuery(**preprocessed)
        print(f"[3] Pydantic 校验通过: city={query.city!r}, date={query.date}, unit={query.unit}")
    except (ValidationError, ValueError) as e:
        print(f"[3] 校验失败: {e}")

---
## 6. mode='wrap' — 高级拦截模式

`mode='wrap'` 允许你**包裹**默认的校验逻辑，可以在前后都插入自定义处理。

In [ ]:
class StrictInt(BaseModel):
    value: int

    @field_validator('value', mode='wrap')
    @classmethod
    def wrap_validate(cls, v, handler):
        print(f"  [wrap 前] 原始值: {v!r} (type={type(v).__name__})")
        if isinstance(v, str) and 'pi' in v.lower():
            v = 3
        result = handler(v)
        print(f"  [wrap 后] 结果值: {result!r} (type={type(result).__name__})")
        if result > 100:
            raise ValueError(f'value 不能超过 100，收到 {result}')
        return result

print("--- 测试 'pi' 字符串 ---")
d1 = StrictInt(value="pi is 3.14")
print(f"结果: {d1.value}")

print("\n--- 测试普通字符串 ---")
d2 = StrictInt(value="42")
print(f"结果: {d2.value}")

print("\n--- 测试超限值 ---")
try:
    StrictInt(value="200")
except ValidationError as e:
    print(f"超限拦截: {e.errors()[0]['msg']}")

In [ ]:
class CrossFieldWrap(BaseModel):
    action: str
    target: str
    force: bool = False

    @model_validator(mode='wrap')
    @classmethod
    def wrap_model_validate(cls, v, handler):
        if isinstance(v, dict):
            action = v.get('action', '')
            if action == 'delete' and not v.get('force'):
                print("  [model wrap] delete 操作未带 force，自动添加 force=False 警告")
        instance = handler(v)
        return instance

ok = CrossFieldWrap(action="delete", target="/tmp/test.txt", force=True)
print(f"delete with force: action={ok.action}, force={ok.force}")

warn = CrossFieldWrap(action="delete", target="/tmp/test.txt")
print(f"delete without force: action={warn.action}, force={warn.force}")

---
## 7. 练习

### 练习 1：API 请求参数拦截
创建一个 `APIRequest` 模型，要求：
- `endpoint`: 必须以 `/api/` 开头，否则自动补全
- `method`: 只允许 GET/POST/PUT/DELETE，大小写不敏感
- `timeout`: 接受字符串/整数，自动转为 float，范围 1~300
- 用 `model_validator` 校验：POST/PUT 才能有 `body`

In [ ]:
# TODO: 在此编写你的 APIRequest 模型
# class APIRequest(BaseModel):
#     ...

# 测试
# r1 = APIRequest(endpoint="v1/users", method="get", timeout="30")
# print(f"合法: endpoint={r1.endpoint}, method={r1.method}, timeout={r1.timeout}")

# try:
#     APIRequest(endpoint="/api/v1/users", method="GET", body={"key": "val"})
# except ValidationError as e:
#     print(f"GET 不应有 body: {e.errors()[0]['msg']}")

### 练习 2：参数名纠错器
扩展 `ParamFixer`，增加以下功能：
- 支持嵌套参数名纠错（如 `config.host` → `server.host`）
- 支持大小写不敏感匹配
- 记录所有纠错日志，最后返回纠错报告

In [ ]:
# TODO: 在此编写你的增强版 ParamFixer
# class EnhancedParamFixer:
#     ...

# 测试
# fixer = EnhancedParamFixer(
#     alias_map={"Location": "city", "TIME": "date"},
#     valid_params={"city", "date"}
# )
# result, report = fixer.fix({"Location": "北京", "TIME": "2024-01-15"})
# print(f"结果: {result}")
# print(f"报告: {report}")

### 练习 3：完整拦截管线
构建一个邮件发送工具的完整拦截管线：
- 参数名纠错：`to` → `recipient`, `subj` → `subject`
- 值纠错：邮箱格式校验、subject 长度限制
- 跨字段校验：`priority=high` 时必须提供 `deadline`

In [ ]:
# TODO: 在此编写你的邮件发送拦截管线
# class EmailRequest(BaseModel):
#     ...

# 测试
# e1 = EmailRequest(recipient="test@example.com", subject="Hello", priority="normal")
# print(f"合法: recipient={e1.recipient}")

# try:
#     EmailRequest(recipient="test@example.com", subject="Urgent", priority="high")
# except ValidationError as e:
#     print(f"high 优先级缺少 deadline: {e.errors()[0]['msg']}")

---
## 总结

| 层级 | 技术 | 解决的问题 |
|------|------|-----------|
| 参数名纠错 | `ParamFixer` 硬编码映射 | 大模型编造不存在的参数名 |
| 参数值纠错 | `FuzzyParamFixer` 模糊匹配 | 大模型拼写错误或使用同义词 |
| 类型清洗 | `@field_validator(mode='before')` | 大模型传错类型（数字→字符串等） |
| 范围验证 | `Field(ge=, le=)` + `@field_validator` | 参数值越界 |
| 跨字段校验 | `@model_validator` | 字段之间有联动关系 |
| 高级拦截 | `mode='wrap'` | 需要同时控制前后处理逻辑 |

**核心原则**：永远不要信任大模型的输出，在工具真正执行前，必须有一层代码对其进行严格的语法与语义检查。